In [1]:
!pip install recbole

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.6 MB/s eta 0:00:00
  Attempting uninstall: colorlog
    Found exi

In [2]:
import tqdm
import polars as pl
import numpy as np
import pandas as pd
import seaborn as sns
import random
import os 
import h5py
import sys
import gc

from matplotlib import pyplot as plt
import pyarrow.parquet as pq

In [3]:
file_path = '/kaggle/input/otto-full-optimized-memory-footprint/train.parquet'
df = pl.read_parquet(file_path)
print(df.head())

shape: (5, 4)
┌─────────┬─────────┬────────────┬──────┐
│ session ┆ aid     ┆ ts         ┆ type │
│ ---     ┆ ---     ┆ ---        ┆ ---  │
│ i32     ┆ i32     ┆ i32        ┆ u8   │
╞═════════╪═════════╪════════════╪══════╡
│ 0       ┆ 1517085 ┆ 1659304800 ┆ 0    │
│ 0       ┆ 1563459 ┆ 1659304904 ┆ 0    │
│ 0       ┆ 1309446 ┆ 1659367439 ┆ 0    │
│ 0       ┆ 16246   ┆ 1659367719 ┆ 0    │
│ 0       ┆ 1781822 ┆ 1659367871 ┆ 0    │
└─────────┴─────────┴────────────┴──────┘


In [4]:
train = pl.read_parquet('/kaggle/input/otto-train-and-test-data-for-local-validation/test.parquet')
test = pl.read_parquet('/kaggle/input/otto-full-optimized-memory-footprint/test.parquet')

df = pl.concat([train, test])

df = df.sort(['session', 'aid', 'ts'])
df = df.with_columns((pl.col('ts') * 1e9).alias('ts'))
df = df.rename({'session': 'session:token', 'aid': 'aid:token', 'ts': 'ts:float'})

In [5]:
print(df.columns)

['session:token', 'aid:token', 'ts:float', 'type']


In [6]:
!mkdir -p /kaggle/working/recbox_data
df[['session:token', 'aid:token', 'ts:float']].write_csv('/kaggle/working/recbox_data/recbox_data.inter', separator='\t')

del df, train, test
gc.collect()

0

In [7]:
import logging
from logging import getLogger
import typing
from typing_extensions import Literal
typing.Literal = Literal
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

from recbole.utils.case_study import full_sort_topk

2025-11-29 03:48:42.177110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764388122.397780      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764388122.458576      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
MAX_ITEM = 50  # SASRec thường cần chuỗi dài hơn một chút so với GRU4Rec

parameter_dict = {
    # === Data ===
    'data_path': '/kaggle/working/',
    'USER_ID_FIELD': 'session',
    'ITEM_ID_FIELD': 'aid',
    'TIME_FIELD': 'ts',
    'user_inter_num_interval': "[5,Inf)",
    'item_inter_num_interval': "[5,Inf)",
    'load_col': {'inter': ['session', 'aid', 'ts']},

    # === Model training ===
    'epochs': 20,                     # SASRec thường cần nhiều epoch hơn GRU4Rec
    'stopping_step': 3,
    'train_batch_size': 256,          # có thể tăng lên nếu GPU đủ mạnh
    'eval_batch_size': 1024,
    'train_neg_sample_args': {
        'distribution': 'uniform',    # uniform hoặc popularity
        'sample_num': 1               # số lượng negative samples mỗi positive
    },   # sử dụng full-sort evaluation
    'learning_rate': 0.001,

    # === Sequence handling ===
    'MAX_ITEM_LIST_LENGTH': MAX_ITEM,
    'MAX_SEQ_LENGTH': MAX_ITEM,       # SASRec cần tham số này
    'hidden_size': 64,                # embedding dimension
    'num_heads': 2,                   # số head trong self-attention
    'num_layers': 2,                  # số layer Transformer
    'hidden_dropout_prob': 0.2,       # dropout cho feed-forward
    'attn_dropout_prob': 0.2,         # dropout cho attention

    # === Evaluation ===
    'eval_args': {
        'split': {'RS': [9, 1, 0]},
        'group_by': 'user',
        'order': 'TO',
        'mode': 'full'
    },

    # === Optimization ===
    'loss_type': 'BPR',               # có thể thử 'CE' (Cross Entropy)

    # === Save model to Kaggle working directory ===
    'checkpoint_dir': '/kaggle/working/',   # folder lưu model
    'save_best': True,                       # lưu model tốt nhất
}

# === Initialize config ===
config = Config(model='SASRec', dataset='recbox_data', config_dict=parameter_dict)

# === Initialize random seed and logger ===
init_seed(config['seed'], config['reproducibility'])
init_logger(config)
logger = getLogger()

# === Create handler để log ra màn hình ===
c_handler = logging.StreamHandler()
c_handler.setLevel(logging.INFO)
logger.addHandler(c_handler)

# === Print config để kiểm tra ===
logger.info(config)


General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /kaggle/working/recbox_data
checkpoint_dir = /kaggle/working/
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 256
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 3
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [9, 1, 0]}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision']
topk = [10]
valid_metric = MRR@10
valid_metric_bigger = True
eval_batch_size = 1024
metric_decimal_place = 4

Dataset Hyper Parameters:
field_sepa

In [9]:
dataset = create_dataset(config)
logger.info(dataset)

/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/usr/local/lib/python3.11/dist-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

In [10]:
# dataset splitting
train_data, valid_data, test_data = data_preparation(config, dataset)

[Training]: train_batch_size = [256] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
[Evaluation]: eval_batch_size = [1024] eval_args: [{'split': {'RS': [9, 1, 0]}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


In [11]:
model = SASRec(config, train_data.dataset).to(config['device'])
logger.info(model)

# trainer loading and initialization
trainer = Trainer(config, model)

# model training
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

SASRec(
  (item_embedding): Embedding(315004, 64, padding_idx=0)
  (position_embedding): Embedding(50, 64)
  (trm_encoder): TransformerEncoder(
    (layer): ModuleList(
      (0-1): 2 x TransformerLayer(
        (multi_head_attention): MultiHeadAttention(
          (query): Linear(in_features=64, out_features=64, bias=True)
          (key): Linear(in_features=64, out_features=64, bias=True)
          (value): Linear(in_features=64, out_features=64, bias=True)
          (softmax): Softmax(dim=-1)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (dense): Linear(in_features=64, out_features=64, bias=True)
          (LayerNorm): LayerNorm((64,), eps=1e-12, elementwise_affine=True)
          (out_dropout): Dropout(p=0.2, inplace=False)
        )
        (feed_forward): FeedForward(
          (dense_1): Linear(in_features=64, out_features=256, bias=True)
          (dense_2): Linear(in_features=256, out_features=64, bias=True)
          (LayerNorm): LayerNorm((64,), eps=1e-12

In [12]:
import gc
del trainer, train_data, valid_data, test_data

In [13]:
gc.collect()

5673